In [2]:
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import time
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import numpy as np

import random

def set_seed(seed):
    """Sets the seed for reproducibility."""
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multi-GPU.
        
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python's random module.
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False

set_seed(1000)

In [147]:
rest_type = 'by-star'

df = pd.read_csv('./data/midwest-pos-neg.csv')

## Text processing

In [148]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text):
    # Some reviews are only numbers (Why? I don't know). Review should always be treated as a string
    text = str(text)
    
    # This appears in some reviews
    text = text.replace('(Translated by Google)', ' ')
    text = text.replace('\n', ' ')

    # Convert to lowercase
    text = text.lower()
    
    # Remove non letter or number entries
    # BERT does OK with numbers is seems
    text = re.sub(r'[^\w\s]', '', text)
    
    return text

# Process the text for better classification
df['text'] = df['text'].apply(preprocess_text)

In [150]:
# Turn ratings into +/-/neutral and do a bit of light processing to the text
# Careful to only run this once

def pnn(n):
    if n in {1,2}: return 0 # negative
    elif n in {4,5}: return 1 # positive

df['pnn'] = df['rating'].apply(pnn)


## Create BERT instance
Skip this step if continuing training

In [151]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer, get_linear_schedule_with_warmup

#model_name = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
model_name = "AnkitAI/reviews-roberta-base-sentiment-analysis"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer and model
model = RobertaForSequenceClassification.from_pretrained(model_name).to(device)
tokenizer = RobertaTokenizer.from_pretrained(model_name)
# Model stats
#print(model)
print('Device: ', device)

Device:  cuda


## Test the output of an untrained model

In [152]:
# Simple test cycle of the untrained model
def predict_sentiment(text):
    inputs = tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    inputs = {'input_ids':input_ids, 'attention_mask':attention_mask}
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        
    return predicted_class, probabilities.tolist()

# Enter your text here
review = 'Really not that bad!'
label, probs = predict_sentiment(review)
meaning = ['negative', 'positive']

print(f'Review: {review}')
print(f'Predicted rating: {label} ({meaning[label]})')
print(f'Probabilities: 0 ({probs[0][0]*100:.2f}%) | 1 ({probs[0][1]*100:.2f}%)')

Review: Really not that bad!
Predicted rating: 1 (positive)
Probabilities: 0 (1.20%) | 1 (98.80%)


## Script for autogenerating plots during training


In [153]:
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

def runtime(start_time):
    end_time = time.time()
    s = end_time - start_time

    h = s // 3600
    s = s - 3600*h

    m = s // 60
    s = s - 60*m

    print(f'Total runtime: {int(h)} hr {int(m)} min {s:.1f} sec')
    return 

def plot_confusion_matrix(cm, classes, title='Confusion Matrix', cmap=plt.cm.Blues):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=classes, yticklabels=classes)
    
    plt.title(title)
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    
    current_time = datetime.datetime.now()
    file_name = f'./sshots/model-{rest_type}-{current_time.strftime("%Y-%m-%d--%H%m")}.png'
    plt.savefig(file_name)
    
    plt.show()
    return

## Training the BERT model begins here

In [154]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

# Create dataset from data
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length #store max length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            text,
            padding="max_length", #important
            truncation=True, #important
            max_length=self.max_length, #important
            return_tensors="pt")
        
        input_ids = inputs['input_ids'].flatten()
        attention_mask = inputs['attention_mask'].flatten()
        label_tensor = torch.tensor(label)

        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': label_tensor}

In [155]:
from sklearn.utils import class_weight

# Ratings are not evenly distributed, this creates class weights
def calculate_class_weights(labels):
    class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return torch.tensor(class_weights, dtype=torch.float)

In [156]:
from sklearn.model_selection import train_test_split

# Export the text and ratings to a list, this part is necessary otherwise pandas keeps the index
X = df['text'].values.tolist()
y = df['pnn'].values.tolist()

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, random_state=666, test_size=.2)

# Batch size set as a power of 2
# monitor RAM use in the terminal with "watch -n5 nvidia-smi"
batch_size = 64
max_length = 128
learning_rate = 2e-5
weight_decay = .01

## Freeze the base BERT parameters and only train the classification layer
for param in model.roberta.parameters():
    param.requires_grad = False

# Create datasets and dataloaders
train_dataset = SentimentDataset(X_train, y_train, tokenizer, max_length=max_length)
val_dataset = SentimentDataset(X_val, y_val, tokenizer, max_length=max_length)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

# Adjust learning rate and weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Calculate class weights of the train set
# class_weights = calculate_class_weights(y_train).to(device)
criterion = nn.CrossEntropyLoss()

#print('Class weights:', class_weights)

In [157]:
def evaluate_model(model, dataloader):
    # Put in evaluation mode
    print('Evaluating...', end='\r')
    model.eval()
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    ## Accuracy and F1 used to determine early exit
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return acc, f1

In [158]:
def early_stopping(model, train_dataloader, val_dataloader, optimizer, epochs=100, patience=5, max_grad_norm=1.0, device='cuda'):
    # Keep track of some data
    start_time = time.time()
    accs, losses, f1s = [], [], []

    # Learning rate scheduler
    num_training_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

    # Set best value as low as possible to start
    best_val_f1 = -np.inf
    patience_counter = 0

    for epoch in range(epochs):
        # Put model in train mode
        model.train()
        train_loss = 0
    
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}"):
            optimizer.zero_grad()
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            ## Evaluate model at the inputs
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    
            ## Evaluate loss and backpropogate
            loss = outputs.loss
            train_loss += loss.item()
            loss.backward()

            ## Clip gradient
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            ## Stepforward the optimizer and scheduler
            optimizer.step()
            scheduler.step()

        # Keep track of loss
        trin_loss = train_loss / len(train_dataloader)
        losses.append(loss)
        
        # Validation cycle
        val_acc, val_f1 = evaluate_model(model=model, dataloader=val_dataloader)
        
        accs.append(val_acc)
        f1s.append(val_f1)

        # Keep track of progress
        print(f'Train loss: {loss:.5f} | Val acc: {100*val_acc:.2f}%, F1 {val_f1:.5f}')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            
            # Save the best model
            torch.save(model.state_dict(), f'./saved-models/bert-pn.pth')
            print('--New best--')
            
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print('Early stopping triggered')
                break 

    # Print total runtime
    runtime(start_time)

    # Load the best model and return it
    model.load_state_dict(torch.load(f'./saved-models/bert-pn.pth'))
    return model

In [159]:
# Train the model here
model = early_stopping(model=model, 
                       train_dataloader=train_dataloader, 
                       val_dataloader=val_dataloader, 
                       optimizer=optimizer, 
                       epochs=100, 
                       patience=3,
                       max_grad_norm=1.0,
                       device=device)

Epoch 1: 100%|██████████████████████████████| 6250/6250 [16:58<00:00,  6.14it/s]


Train loss: 0.23323 | Val acc: 97.06%, F1 0.97060
--New best--


Epoch 2: 100%|██████████████████████████████| 6250/6250 [16:40<00:00,  6.25it/s]


Train loss: 0.08514 | Val acc: 97.14%, F1 0.97136
--New best--


Epoch 3: 100%|██████████████████████████████| 6250/6250 [16:44<00:00,  6.22it/s]


...

Epoch 8: 100%|██████████████████████████████| 6250/6250 [16:42<00:00,  6.23it/s]


Train loss: 0.04025 | Val acc: 97.12%, F1 0.97119


Epoch 9: 100%|██████████████████████████████| 6250/6250 [16:42<00:00,  6.23it/s]


Train loss: 0.16765 | Val acc: 97.09%, F1 0.97090
Early stopping triggered
Total runtime: 3 hr 6 min 10.6 sec


/tmp/ipykernel_1183437/2902697126.py:72: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'./saved-models/bert-pn.pth'))


## Making predictions on 2,3 and 4 star reviews

In [160]:
df_predict = pd.read_csv('./data/midwest-234-test.csv')

df_predict['text'] = df_predict['text'].apply(preprocess_text)

ds_predict = SentimentDataset(df_predict['text'].values.tolist(), [0]*len(df_predict), tokenizer, max_length=max_length)
dl_predict = DataLoader(ds_predict, batch_size=batch_size)


In [161]:
def make_prediction(model, dataloader):
    model.eval()
    
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1).tolist()
            all_probs.extend(probs)

    return all_probs

def predict_sent(probs, pos_thresh=.8, neg_thresh=.8):
    if probs[1] > pos_thresh:
        return 'positive'

    elif probs[0] > neg_thresh:
        return 'negative'

    else:
        return 'mixed'

In [162]:
probs = make_prediction(model, dl_predict)
df_predict['probs'] = probs



Evaluating 300000 predictions...: 100%|█████| 4688/4688 [11:57<00:00,  6.53it/s]


In [173]:
df_predict.sample(20)

,rating,text,type,probs,preds_0.5,preds_0.55,preds_0.6,preds_0.65,preds_0.7,preds_0.75,preds_0.8,preds_0.8500000000000001,preds_0.9,preds_0.95
10523,2,bad service,Takeout Restaurant,"[0.9961615800857544, 0.003838358912616968]",negative,negative,negative,negative,negative,negative,negative,negative,negative,negative
49514,2,the quality of food is not the same,Restaurant,"[0.9943457245826721, 0.005654216278344393]",negative,negative,negative,negative,negative,negative,negative,negative,negative,negative
248900,4,great pizza at great prices sunday only half off,Restaurant,"[0.0018958629807457328, 0.9981040954589844]",positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
188430,3,will go back if i absolutely have to,Restaurant,"[0.9978125095367432, 0.0021874154917895794]",negative,negative,negative,negative,negative,negative,negative,negative,negative,negative
174159,3,food and service is ok nothing special the reg...,Hamburger restaurant,"[0.7717747688293457, 0.22822530567646027]",negative,negative,negative,negative,negative,negative,mixed,mixed,mixed,mixed
268968,4,did not seat us at the best place in the resta...,Sushi restaurant,"[0.45603543519973755, 0.5439645648002625]",positive,mixed,mixed,mixed,mixed,mixed,mixed,mixed,mixed,mixed
204582,4,food was fresh,Traditional American restaurant,"[0.019165953621268272, 0.9808340668678284]",positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
85369,2,not the best culvers,Restaurant,"[0.9746752977371216, 0.02532474510371685]",negative,negative,negative,negative,negative,negative,negative,negative,negative,negative
262337,4,great place to eat,Southern restaurant (US),"[0.01930025778710842, 0.9806997179985046]",positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
223323,4,good cheap road food hit this place up leave a...,American restaurant,"[0.003816690994426608, 0.9961833357810974]",positive,positive,positive,positive,positive,positive,positive,positive,positive,positive


In [169]:
df_predict.to_csv('predicted-sentiment.csv', index=False)

In [144]:
print('2-star')
print(df_predict[df_predict['rating']==2].groupby('preds').count())
print()

print('3-star')
print(df_predict[df_predict['rating']==3].groupby('preds').count())
print()

print('4-star')
print(df_predict[df_predict['rating']==4].groupby('preds').count())

2-star
          rating  text  type  probs
preds                              
mixed        305   305   305    305
negative    2909  2909  2909   2909
positive     119   119   119    119

3-star
          rating  text  type  probs
preds                              
mixed        837   837   837    837
negative    1665  1665  1665   1665
positive     853   853   853    853

4-star
          rating  text  type  probs
preds                              
mixed        364   364   364    364
negative     233   233   233    233
positive    2715  2715  2715   2715


In [ ]:
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)